In [1]:
# Cell 1: Imports
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification, Trainer, TrainingArguments
from transformers import DataCollatorWithPadding
import torch.nn as nn
from torch.utils.data import Dataset
import pandas as pd
import shap
import numpy as np
from datasets import Dataset as HFDataset
from pathlib import Path
import torch

BASE_DIR = Path.cwd().resolve()
if BASE_DIR.name == "notebooks":
    BASE_DIR = BASE_DIR.parent

TEXT_DIR = BASE_DIR / "data" / "text"



train_df = pd.read_csv(TEXT_DIR / "train.csv")
val_df = pd.read_csv(TEXT_DIR / "val.csv")
test_df = pd.read_csv(TEXT_DIR / "test.csv")


In [2]:
# Cell 2: Text Dataset Class
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df["text"].values
        self.labels = df["label"].values
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, truncation=True, padding='max_length',
            max_length=self.max_len, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(self.labels[idx])
        }


In [3]:
# Cell 3: Model + Training

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=2
)

# Optional DirectML (only if torch_directml works)
try:
    import torch_directml
    device = torch_directml.device()
    model.to(device)
    print("✅ Using DirectML")
except Exception as e:
    print("⚠️ DirectML not available, using CPU:", e)

train_dataset = TextDataset(train_df, tokenizer)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

args = TrainingArguments(
    output_dir='./text_results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)
trainer.train()

model.save_pretrained("models/text_distilbert")
tokenizer.save_pretrained("models/text_distilbert")
print("✅ Text model trained")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Using DirectML


Step,Training Loss
500,0.231300
1000,0.116800
1500,0.117200
2000,0.112300
2500,0.113700
3000,0.107500
3500,0.103500
4000,0.100400
4500,0.105300
5000,0.093700


✅ Text model trained


In [7]:
# Cell 4: SHAP Explainability (no retrain)
import shap
from transformers import pipeline

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    return_all_scores=True
)

# explain class index 1 (fake)
explainer = shap.Explainer(clf)
texts = ["sample fake text", "sample real text"]
shap_values = explainer(texts, fixed_context=1)

shap.plots.text(shap_values[0])

Device set to use cpu
c:\Users\hp\anaconda3\Lib\site-packages\transformers\pipelines\text_classification.py:106: UserWarning: `return_all_scores` is now deprecated,  if want a similar functionality use `top_k=None` instead of `return_all_scores=True` or `top_k=1` instead of `return_all_scores=False`.
  warnings.warn(


In [8]:
print(shap_values)

.values =
array([[[-0.01063642,  0.01063647],
        [-0.01395972,  0.01395973],
        [-0.01672417,  0.01672417],
        [-0.01522788,  0.01522786],
        [-0.01140983,  0.01140982]],

       [[-0.00137187,  0.00137181],
        [-0.00711839,  0.00711841],
        [ 0.00028263, -0.00028266],
        [ 0.00441449, -0.00441445],
        [ 0.00314082, -0.00314085]]])

.base_values =
array([[0.06835999, 0.93164003],
       [0.00126583, 0.99873418]])

.data =
(array(['', ' sample', ' fake', ' text', ''], dtype=object), array(['', ' sample', ' real', ' text', ''], dtype=object))


In [9]:
model.save_pretrained("models/text_distilbert")
tokenizer.save_pretrained("models/text_distilbert")

('models/text_distilbert\\tokenizer_config.json',
 'models/text_distilbert\\special_tokens_map.json',
 'models/text_distilbert\\vocab.txt',
 'models/text_distilbert\\added_tokens.json')